## Reading 33-2 - Performance Correctness: Cache, GEMM, and BLAS

By the end of this reading, you will learn:
<ol>
    <li>How a two-dimensional matrix is laid out in memory</li>
    <li>Why the <i>order</i> you visit memory can change run time by 10x with no change in the answer</li>
    <li>How to time code correctly in C</li>
    <li>What GEMM and BLAS are, and why every processor on a spacecraft is built to accelerate them</li>
</ol>

### A Late Answer Is a Wrong Answer

In Reading 32-1, we saw that flight software must deliver the right answer <i>on time</i>. A guidance computer that computes the correct attitude 200 ms after it was needed has failed. That makes performance part of correctness, not just a nice extra.

### How a Matrix Lives in Memory

Memory is one long line of addresses. A matrix has rows and columns. C stores a two-dimensional array in <b>row-major</b> order: all of row 0, then all of row 1, and so on. If we allocate an <code>N x N</code> matrix with <code>malloc</code>, we find element <code>[row][col]</code> with pointer arithmetic:

    mat[row * N + col]

Moving one column to the right moves 8 bytes (one <code>double</code>). Moving one row down moves <code>N * 8</code> bytes.

### Same Answer, Ten Times Slower - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec33/row_vs_col.c">row_vs_col.c</a>

This program sums every element of a 2048 x 2048 matrix twice. The only difference is the order of the loops:

    /* Row by row: the next element is the next address in memory */
    for (int row = 0; row < N; row++) {
    	for (int col = 0; col < N; col++) {
    		sum += mat[row * N + col];
    	}
    }

    /* Column by column: the next element is N doubles away */
    for (int col = 0; col < N; col++) {
    	for (int row = 0; row < N; row++) {
    		sum += mat[row * N + col];
    	}
    }

    $ make row_vs_col
    Run 1: rows 0.0037 s (sum 4194304)   cols 0.0330 s (sum 4194304)   ratio 9.0x
    Run 2: rows 0.0031 s (sum 4194304)   cols 0.0322 s (sum 4194304)   ratio 10.4x
    Run 3: rows 0.0033 s (sum 4194304)   cols 0.0343 s (sum 4194304)   ratio 10.3x

Your numbers on <code>student13</code> will be different, but the ratio will be large.

The reason is the <b>cache</b>. The processor never reads a single <code>double</code> from main memory. It reads a whole <i>cache line</i>, typically 64 bytes, or 8 <code>double</code> values, and keeps it in a small, fast memory close to the processor.
<ul>
    <li><b>Row order:</b> one trip to main memory brings in 8 values, and we use all 8</li>
    <li><b>Column order:</b> one trip brings in 8 values, we use 1, and by the time we come back for the others, they have been evicted</li>
</ul>

Same arithmetic. Same answer. Ten times the cost. A matrix multiply has three nested loops, so the loop order matters even more there, as you will measure in Homework 9.

### Timing Code Correctly

The example uses this function to read the clock:

    /* clock_gettime is part of POSIX, not ISO C, so -std=c2x hides it unless we ask */
    #define _POSIX_C_SOURCE 199309L

    double now_seconds(void)
    {
    	struct timespec ts;
    	clock_gettime(CLOCK_MONOTONIC, &ts);
    	return (double)ts.tv_sec + (double)ts.tv_nsec * 1.0e-9;
    }

Three details matter:
<ul>
    <li><b>The <code>#define</code> must come before every <code>#include</code>.</b> Our flag <code>-std=c2x</code> means "strict ISO C only," and <code>clock_gettime</code> comes from the POSIX standard instead. Without the define, you get <code>implicit declaration of function 'clock_gettime'</code>.</li>
    <li><b>Use <code>CLOCK_MONOTONIC</code>, not the wall clock.</b> The wall clock can jump when the system synchronizes its time. A monotonic clock only moves forward, which is exactly the rule flight software follows for onboard time.</li>
    <li><b>Run it more than once.</b> Other programs share the machine, so any single measurement can be noisy. Repeat the measurement and report the <i>median</i>, which ignores a single unusually slow run.</li>
</ul>

### GEMM: The Kernel Every Engine Accelerates

The most important operation in numerical computing is <b>GEMM</b>, General Matrix Multiply:

<p align="center">C = &alpha;(A &times; B) + &beta;C</p>

<ul>
    <li><code>A</code> is M x K, <code>B</code> is K x N, and <code>C</code> is M x N</li>
    <li>&alpha; and &beta; are scalars. With &alpha; = 1 and &beta; = 0, GEMM is an ordinary matrix multiply</li>
</ul>

GEMM appears all over a spacecraft: rotating vectors between coordinate frames, the Kalman filter that estimates the spacecraft's position (Lecture 38), image processing, and neural-network inference. Every specialized processor on a modern spacecraft, whether GPU, NPU, or DSP, is at its core a faster way to compute this one operation.

<b>Counting the work:</b> Multiplying two m x m matrices takes m<sup>3</sup> multiplications and m<sup>3</sup> additions, so about 2m<sup>3</sup> floating-point operations (FLOPs). Performance is reported in billions of FLOPs per second, or <b>GFLOP/s</b>:

<p align="center">GFLOP/s = 2m<sup>3</sup> / (time in seconds &times; 10<sup>9</sup>)</p>

For example, multiplying two 1000 x 1000 matrices in 0.5 seconds is 2 &times; 10<sup>9</sup> / 0.5 / 10<sup>9</sup> = 4 GFLOP/s.

### BLAS: One Interface, Many Implementations - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec33/blas_dgemm.c">blas_dgemm.c</a>

<b>BLAS</b> (Basic Linear Algebra Subprograms) is a standard <i>interface</i> for vector and matrix operations that dates back to the late 1970s. The interface stays the same, and each hardware vendor supplies an implementation tuned for its chip:
<ul>
    <li><b>Reference BLAS:</b> correct and portable, but not optimized</li>
    <li><b>OpenBLAS:</b> an open-source implementation tuned for common CPUs, installed on <code>student13</code></li>
    <li><b>Vendor libraries</b> for specific DSPs and accelerators</li>
</ul>

This is how flight software stays <i>portable</i>: the mission code calls the BLAS interface, and when the hardware changes, you link a different library instead of rewriting the mission code.

The double-precision GEMM function is <code>cblas_dgemm</code> (the <code>d</code> is for <code>double</code>):

    cblas_dgemm(CblasRowMajor,	/* C stores 2D arrays row by row */
    	    CblasNoTrans,	/* use A as-is (not transposed) */
    	    CblasNoTrans,	/* use B as-is */
    	    2, 2, 2,		/* M, N, K: A is MxK, B is KxN, C is MxN */
    	    1.0,		/* alpha */
    	    &a[0][0], 2,	/* A and its row length */
    	    &b[0][0], 2,	/* B and its row length */
    	    0.0,		/* beta */
    	    &c[0][0], 2);	/* C and its row length */

    $ make blas_dgemm
      19.0   22.0
      43.0   50.0

To use BLAS on <code>student13</code>, include <code>&lt;cblas.h&gt;</code> and link the library at the <i>end</i> of the compile command, after your source file:

    LIBS := -l:libopenblas.so.0 -lm

The unusual <code>-l:</code> syntax asks for the library file by its exact name, which is how this library is installed on <code>student13</code>.

### Same GEMM, Different Answers

Floating-point addition is not associative. Consider:
<ul>
    <li><code>(1.0e16 + 1.0) - 1.0e16</code> evaluates to <code>0.0</code>, because the 1.0 is smaller than the spacing between <code>double</code> values near 10<sup>16</sup> and is rounded away</li>
    <li><code>(1.0e16 - 1.0e16) + 1.0</code> evaluates to <code>1.0</code></li>
</ul>

An optimized BLAS adds the terms of each sum in a different order than your three nested loops do, so the two results may differ in the last few bits. An NPU working in 8-bit integers will differ by much more. Neither answer is "the" answer.

This leads to a rule you will use in Homework 9: <b>never compare floating-point results with <code>==</code>.</b> Validate an optimized result against a reference implementation, and check that the difference is within a stated tolerance.

### <font color = "red">Class Introduction Question #4 - C stores matrices in row-major order. Explain why summing by columns is so much slower than summing by rows, even though the arithmetic is identical.</font>

### <font color = "red">Class Introduction Question #5 - Why should you use <code>CLOCK_MONOTONIC</code> and report the median of several runs when timing code?</font>

### <font color = "red">Class Introduction Question #6 - What is the difference between the BLAS <i>interface</i> and a BLAS <i>implementation</i>, and why does that difference matter for flight software that may run on different processors?</font>

### In class, we will measure the cost of memory order in a matrix multiply, and see how far a hand-written loop is from BLAS.